In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import pandas as pd
from pathlib import Path
import seaborn as sns

In [ ]:
results_dir = Path(".") / "results"
records = []

type_dict = {
    "SEQ-FINDPIVOT": "Sequential with findPivot",
    "SEQ-FINDPIVOTSAFELY": "Sequential with findPivotSafely",
    "OPENMP-MAPREDUCE": "OpenMP as MapReduce",
    "OPENMP-MUTEX": "OpenMP with mutexes",
    "STDTHREAD": "std::thread",
    "CUDA": "CUDA"
}

for path in sorted(results_dir.glob("*.txt")):
    type_value, vertices_str = path.stem.split("_", 1)
    vertices_number = int(vertices_str)
    
    with path.open("r", encoding="utf-8") as file:
        for measure_number, line in enumerate(file, start=1):
            overallTime, edgePhaseTime, _ = line.split(",")
            records.append({
                "type": type_dict.get(type_value),
                "vertices_number": vertices_number,
                "measure_number": measure_number,
                "overallTime": int(overallTime.strip()),
                "edgePhaseTime": int(edgePhaseTime.strip())
            })

results_df = pd.DataFrame(records)
results_df.head()


In [ ]:
# Convert microseconds to seconds
results_df["overallTime"] /= 1_000_000
results_df["edgePhaseTime"] /= 1_000_000

results_df.head()

In [ ]:
# Remove outliers using "overallTime" field
grouped = results_df.groupby(["type", "vertices_number"])["overallTime"]

q1 = grouped.transform(lambda s: s.quantile(0.25))
q3 = grouped.transform(lambda s: s.quantile(0.75))
iqr = q3 - q1

lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

filtered_df = results_df[results_df["overallTime"].between(lower, upper)]

filtered_df.head()

In [ ]:
overall_avg_df = (
    filtered_df
    .groupby(["type", "vertices_number"], as_index=False)["overallTime"]
    .mean()
    .rename(columns={"overallTime": "avg_value"})
)

overall_avg_df.head()

In [ ]:
edgephase_avg_df = (
    filtered_df
    .groupby(["type", "vertices_number"], as_index=False)["edgePhaseTime"]
    .mean()
    .rename(columns={"edgePhaseTime": "avg_value"})
)

edgephase_avg_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for typ, group in overall_avg_df.groupby("type"):
    group_sorted = group.sort_values("vertices_number")
    ax.plot(
        group_sorted["vertices_number"],
        group_sorted["avg_value"],
        marker="o",
        label=typ,
    )

ax.set_xlabel("Vertices number in graph")
ax.set_ylabel("Average execution time (in seconds)")
ax.set_title("Overall Execution Time by Implementation")
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x/1_000_000:.1f}M'))
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for typ, group in edgephase_avg_df.groupby("type"):
    group_sorted = group.sort_values("vertices_number")
    ax.plot(
        group_sorted["vertices_number"],
        group_sorted["avg_value"],
        marker="o",
        label=typ,
    )

ax.set_xlabel("Vertices number in graph")
ax.set_ylabel("Average execution time (in seconds)")
ax.set_title("Edge Phase Execution Time by Implementation")
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{x/1_000_000:.1f}M'))
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
subset_df = results_df[results_df["vertices_number"] == 1_000_000]

subset_df.head()

In [ ]:
order = type_dict.values()

In [ ]:
values = [subset_df[subset_df["type"] == typ]["overallTime"] for typ in order]

fig, ax = plt.subplots(figsize=(6, 4))
bp = ax.boxplot(
    values,
    tick_labels=order,
    patch_artist=True,
    showfliers=False,
    whiskerprops=dict(color="navy"),
    capprops=dict(color="navy"),
    medianprops=dict(color="black"),
)

ax.set_xlabel("Implementations")
ax.set_ylabel("Execution time (in seconds)")
ax.set_title("Overall execution Time Distribution for 1M Vertices graph")
ax.grid(axis="y")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
sns.violinplot(
    data=subset_df,
    x="type",
    y="overallTime",
    order=order,
    ax=ax,
)

ax.set_xlabel("Implementations")
ax.set_ylabel("Execution time (in seconds)")
ax.set_title("Overall execution Time Distribution for 1M Vertices Graph")
ax.grid(axis="y")
plt.tight_layout()
plt.show()

In [ ]:
# Stacked bar chart for overall time and edge phase for 1M vertices
avg_overall = overall_avg_df[overall_avg_df["vertices_number"] == 1_000_000].set_index("type")
avg_edge = edgephase_avg_df[edgephase_avg_df["vertices_number"] == 1_000_000].set_index("type")

avg_combined = avg_overall.join(avg_edge, how="inner", lsuffix="_overall", rsuffix="_edge")
avg_combined["other"] = avg_combined["avg_value_overall"] - avg_combined["avg_value_edge"]

order = list(type_dict.values())
avg_combined = avg_combined.reindex(order)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(order, avg_combined["other"], label='Other phases', color='skyblue')
ax.bar(order, avg_combined["avg_value_edge"], bottom=avg_combined["other"], label='Edge phase', color='orange')

ax.set_xlabel('Implementations')
ax.set_ylabel('Average execution time (in seconds)')
ax.set_title('Execution time by Phase for the Graph with 1 Million Vertices')
ax.legend()
ax.grid(axis='y')
plt.tight_layout()
plt.show()

In [ ]:
df_loc = pd.DataFrame({
    "implementation": [
        type_dict.get("OPENMP-MAPREDUCE"),
        type_dict.get("STDTHREAD"),
        type_dict.get("CUDA"),
    ],
    "physical_lines": [148, 190, 235],
    "LOC_total": [120, 149, 191],
    "LOC_parallel": [5, 12, 28],
    "difficulty_score": [1, 3, 8]
    
})

df_loc.head()

In [ ]:
df_loc["parallelization_effort"] = 100 * df_loc["LOC_parallel"] / df_loc["LOC_total"]

df_loc.head()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(df_loc["implementation"], df_loc["parallelization_effort"], color="tab:blue")

for idx, effort in enumerate(df_loc["parallelization_effort"]):
    ax.text(idx, effort + 1, f"{effort:.1f}%", ha="center", va="bottom")

ax.set_ylabel("Parallelization effort (in %)")
ax.set_title("Parallelization Effort by Implementation")
ax.set_ylim(0, df_loc["parallelization_effort"].max() * 1.2)
ax.grid(axis="y", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(df_loc["implementation"], df_loc["difficulty_score"], color="tab:blue")

for idx, difficulty in enumerate(df_loc["difficulty_score"]):
    ax.text(idx, difficulty + 1, f"{difficulty}", ha="center", va="bottom")

ax.set_ylabel("Difficulty score")
ax.set_title("Difficulty Score by Implementation")
ax.set_ylim(0, df_loc["difficulty_score"].max() * 1.2)
ax.grid(axis="y", alpha=0.6)
plt.tight_layout()
plt.show()